# K-Phase: Knowledge Transfer (Deployment)

## Ziel
Die K-Phase überführt das ausgewählte Modell aus der Analyse in eine betreibbare Lösung mit dokumentierter Übergabe.


## Deployment-Strategie
### Option 1: Embedded Deployment (Streamlit)
- Modell wird direkt in der App geladen.
- Vorteil: schnell umsetzbar, geringe Infrastruktur.
- Nachteil: begrenzte Skalierbarkeit.

### Option 2: Model-as-a-Service (FastAPI)
- Modell als separater Inferenzservice (`/predict`).
- Vorteil: entkoppelte Architektur, bessere Skalierbarkeit und Wiederverwendbarkeit.
- Nachteil: höherer Betriebs- und Monitoringaufwand.

## Empfohlenes Zielbild
Kurzfristig Embedded für Demo/PoC, mittelfristig Migration auf FastAPI-Inferenzservice mit Streamlit als Client.


## K-Check: Deployment und Repository-Übergabe

- **Die fertige Streamlit App**
  Übergabe erfolgt über die App im Repository (`src/traffic_app`) inklusive Modellartefakt und Reports.
- **Das GitHub Repo**
  Zentrale Projektquelle: https://github.com/lukasp1209/Traffic-Prediction-Optimization
- **Wissensbasis im Repo**
  Das Repo enthält reproduzierbare Notebook-Phasen (Q/U/A/C/K), Exportartefakte unter `reports/` sowie Dokumentation unter `docs/`.


In [1]:
from pathlib import Path
import json
import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORT_DIR = repo_root / "reports"
MODEL_DIR = REPORT_DIR / "models"
summary_path = REPORT_DIR / "model_summary.json"
results_path = REPORT_DIR / "model_results.csv"

missing = [str(p) for p in [summary_path, results_path] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "A-Phase-Ergebnisse fehlen. Bitte zuerst A-Phase_Algorithms.ipynb ausf?hren. Fehlend: " + ", ".join(missing)
    )

summary = json.loads(summary_path.read_text(encoding="utf-8"))
results = pd.read_csv(results_path).sort_values("RMSE").reset_index(drop=True)
best_model = summary["best_model"]

artifact_candidates = list(MODEL_DIR.glob(f"best_model_{best_model}.pkl"))
artifact_exists = len(artifact_candidates) > 0

portfolio_summary = {
    "Project": "Traffic Prediction & Optimization",
    "Methodology": "QUA3CK",
    "Best_Algorithm": best_model,
    "Dataset_Rows": int(summary["dataset_rows"]),
    "Train_Rows": int(summary["n_train"]),
    "Test_Rows": int(summary["n_test"]),
    "Top_Result": results.iloc[0].to_dict(),
    "Model_Artifact_Available": artifact_exists,
    "Deployment_Target": "FastAPI Inference Service + Streamlit UI",
    "Next_Steps": [
        "API endpoint /predict mit Request-Validierung",
        "Containerized deployment via docker compose",
        "Monitoring: Latenz, Fehlerrate, RMSE-Drift",
        "Regelmäßiges Re-Training mit aktuellen Verkehrsdaten",
    ],
}

print("K-Phase Portfolio Summary")
print("=" * 60)
for key, value in portfolio_summary.items():
    print(f"{key}: {value}")


K-Phase Portfolio Summary
Project: Traffic Prediction & Optimization
Methodology: QUA3CK
Best_Algorithm: RandomForest
Dataset_Rows: 8734
Train_Rows: 6987
Test_Rows: 1747
Top_Result: {'Model': 'RandomForest', 'MAE': 4874.081155802304, 'RMSE': 6807.078467974301, 'R2': 0.9597666764999648}
Model_Artifact_Available: True
Deployment_Target: FastAPI Inference Service + Streamlit UI
Next_Steps: ['API endpoint /predict mit Request-Validierung', 'Containerized deployment via docker compose', 'Monitoring: Latenz, Fehlerrate, RMSE-Drift', 'Regelmäßiges Re-Training mit aktuellen Verkehrsdaten']


In [2]:
from textwrap import dedent

model_card = dedent(f"""
# Model Card (Kurzfassung)

## Modell
- Name: {summary['best_model']}
- Anwendungsfall: Kurzfristige Verkehrsprognose (Stundenebene)

## Trainingskontext
- Datensätze: {summary['dataset_rows']}
- Train/Test: {summary['n_train']} / {summary['n_test']} (chronologisch)

## Kernmetriken (Test)
- MAE: {results.iloc[0]['MAE']:.3f}
- RMSE: {results.iloc[0]['RMSE']:.3f}
- R2: {results.iloc[0]['R2']:.4f}

## Grenzen
- Synthetische Anteile möglich (abhängig von U-Phase-Konfiguration)
- Externe Ereignisse (Unfälle, Großevents, Baustellen) nur begrenzt abgebildet
- Regelmäßige Validierung auf Live-Daten erforderlich
""").strip()

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
model_card_path = repo_root / "docs" / "guides" / "model-card.md"

# Fehlende Verzeichnisse robust anlegen
model_card_path.parent.mkdir(parents=True, exist_ok=True)

# Datei schreiben
model_card_path.write_text(model_card, encoding="utf-8")

print(f"Model Card geschrieben: {model_card_path.resolve()}")
print(f"Aktueller Arbeitsordner: {Path.cwd()}")


Model Card geschrieben: E:\Traffic-Prediction-Optimization_WIP\docs\guides\model-card.md
Aktueller Arbeitsordner: E:\Traffic-Prediction-Optimization_WIP\notebooks


## Ergebnis der K-Phase
- Übergabefähige Zusammenfassung wurde erzeugt.
- Modellartefakt und Leistungsdaten sind dokumentiert und maschinenlesbar abgelegt.
- Ein operativer Deployment-Pfad ist definiert (Streamlit + Docker + Repository-Übergabe über GitHub).
